## Spark Aggregate Functions

#### Funciones simples de agregación

In [0]:
%run "../includes/configuration"

In [0]:
movies_df = spark.read.parquet(f"{silver_folder_path}/movies")

In [0]:
from pyspark.sql.functions import count, countDistinct, sum

In [0]:
movies_df.select(count("*")).show()

+--------+
|count(1)|
+--------+
|    4725|
+--------+



In [0]:
movies_df.select(count("year_release_date")).show()

+------------------------+
|count(year_release_date)|
+------------------------+
|                    4426|
+------------------------+



In [0]:
movies_df.select(countDistinct("year_release_date")).show()

+---------------------------------+
|count(DISTINCT year_release_date)|
+---------------------------------+
|                               90|
+---------------------------------+



In [0]:
movies_df.select(sum("budget")).display()

sum(budget)
140606455289


In [0]:
movies_df.filter("year_release_date = 2016")\
        .select(sum("budget"), count("movie_id"))\
        .withColumnRenamed("sum(budget)", "total_budget")\
        .withColumnRenamed("count(movie_id)", "count_movies")\
        .display()

total_budget,count_movies
4290150000,94


## Group By

In [0]:
from pyspark.sql.functions import count, countDistinct, sum, max, min, avg

In [0]:
movie_group_by_df = movies_df \
    .groupBy("year_release_date")\
    .agg(
        sum("budget").alias("total_budget"),
        avg("budget").alias("avg_budget"),
        max("budget").alias("max_budget"),
        min("budget").alias("min_budget"),
        count("movie_id")
    )

In [0]:
display(movie_group_by_df)

year_release_date,total_budget,avg_budget,max_budget,min_budget,count(movie_id)
1959,11383848,3794616,5000000,2883848,3
1990,656225000,26249000,70000000,225000,25
1975,26700000,4450000,11000000,300000,6
1977,136110000,9074000,22000000,10000,15
2003,5173137898,3.359180453246753E7,200000000,46000,154
2007,5711147510,3.1379931373626374E7,300000000,15000,182
1974,33085000,4135625,13000000,85000,8
2015,6508097363,3.2869178601010103E7,280000000,50000,198
1927,92620000,92620000,92620000,92620000,1
1955,3200000,1600000,2000000,1200000,2


### Window Functions

In [0]:
from pyspark.sql.functions import rank, desc, dense_rank
from pyspark.sql.window import Window

In [0]:
movie_rank = Window.partitionBy("year_release_date").orderBy(desc("budget"))
movie_dense_rank = Window.partitionBy("year_release_date").orderBy(desc("budget"))
movies_df.select("title", "budget", "year_release_date")\
    .filter("year_release_date is not null")\
    .withColumn("rank", rank().over(movie_rank))\
    .withColumn("dense_rank", dense_rank().over(movie_dense_rank))\
    .display()